In [0]:
%run ../functions/functions

In [0]:

database_name = "dimensao"
table_name = "dm_municipio"
target_path = f"{database_name}.{table_name}"
pk = "PK_MUNICIPIO"

In [0]:
#Bases utilizadas no relacionamento
#base de comex
silver_path_uf_mun = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/UF_MUN/"
silver_path_uf = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/UF/"

#base de cnpj
silver_path_municipios_cnpj = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/MUNICIPIOS_CONSOLIDADA/"

#base de cnpj
silver_path_estabelecimento = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/ESTABELECIMENTOS_CONSOLIDADA/"




In [0]:
df_uf_mun= spark.read.format("delta").load(silver_path_uf_mun)

df_municipios_cnpj= spark.read.format("delta").load(silver_path_municipios_cnpj)

df_municipios_estabelecimento= spark.read.format("delta").load(silver_path_estabelecimento)

df_uf= spark.read.format("delta").load(silver_path_uf)

In [0]:
df_uf_mun.createOrReplaceTempView("df_uf_mun")
df_municipios_cnpj.createOrReplaceTempView("df_municipios_cnpj")

df_municipios_estabelecimento.createOrReplaceTempView("df_municipios_estabelecimento")

df_uf.createOrReplaceTempView("df_uf")

In [0]:
query = """
WITH mun_uf AS (
  SELECT DISTINCT
    try_cast(municipio as int) as codigo_municipio,
    upper(uf) as uf
  FROM df_municipios_estabelecimento
),

cnpj_com_uf AS (
  SELECT
    cn.codigo_municipio,
    -- Aplicando REPLACEs manuais para converter os nomes da base CNPJ para o padrão da base df_uf_mun
    CASE 
      WHEN upper(cn.descricao_municipio) = 'PINGO D''AGUA' THEN 'PINGO-D''AGUA'
      WHEN upper(cn.descricao_municipio) = 'AMPARO DA SERRA' THEN 'AMPARO DO SERRA'
      WHEN upper(cn.descricao_municipio) = 'FLORIANO' THEN 'FLORINIA'
      WHEN upper(cn.descricao_municipio) = 'BARAO DO MONTE ALTO' THEN 'BARAO DE MONTE ALTO'
      WHEN upper(cn.descricao_municipio) = 'EMBU DAS ARTES' THEN 'EMBU'
      WHEN upper(cn.descricao_municipio) = 'MOGI MIRIM' THEN 'MOGI-MIRIM'
      WHEN upper(cn.descricao_municipio) = 'PICARRA' THEN 'PICARRAS'
      WHEN upper(cn.descricao_municipio) = 'IGUARACY' THEN 'IGUARACI'
      WHEN upper(cn.descricao_municipio) = 'ENTRE IJUIS' THEN 'ENTRE-IJUÍS'
      WHEN upper(cn.descricao_municipio) = 'SAO LUIZ DO PARAITINGA' THEN 'SAO LUIS DO PARAITINGA'
      WHEN upper(cn.descricao_municipio) = 'PASSA VINTE' THEN 'PASSA-VINTE'
      WHEN upper(cn.descricao_municipio) = 'SAO VICENTE DO SERIDO' THEN 'SERIDO'
      WHEN upper(cn.descricao_municipio) = 'LAGOA DE ITAENGA' THEN 'LAGOA DO ITAENGA'
      WHEN upper(cn.descricao_municipio) = 'GRACCHO CARDOSO' THEN 'GRACHO CARDOSO'
      WHEN upper(cn.descricao_municipio) = 'PINDARE MIRIM' THEN 'PINDARE-MIRIM'
      WHEN upper(cn.descricao_municipio) = 'SAO FRANCISCO DE ASSIS DO PIAUI' THEN 'SAO FRANCISCO DE ASSIS PIAUI'
      WHEN upper(cn.descricao_municipio) = 'POXOREU' THEN 'POXOREO'
      WHEN upper(cn.descricao_municipio) = 'BELEM DO SAO FRANCISCO' THEN 'BELEM DE SAO FRANCISCO'
      WHEN upper(cn.descricao_municipio) = 'ITAPAJE' THEN 'ITAPAGE'
      WHEN upper(cn.descricao_municipio) = 'OLHO D''AGUA DO BORGES' THEN 'OLHO-D''AGUA DO BORGES'
      WHEN upper(cn.descricao_municipio) = 'SAO TOME DAS LETRAS' THEN 'SAO THOME DAS LETRAS'
      ELSE upper(cn.descricao_municipio)
    END as municipio,
    mu.uf
  FROM df_municipios_cnpj cn
  LEFT JOIN mun_uf mu
    ON cast(cn.codigo_municipio as int) = cast(mu.codigo_municipio as int)
)

SELECT DISTINCT
  m.CO_MUN_GEO,
  m.NO_MUN,
  m.SG_UF,
  cn.codigo_municipio,
  cn.municipio,
  cn.uf,
  NO_UF,
  NO_REGIAO
FROM df_uf_mun m
 JOIN cnpj_com_uf cn
  ON upper(m.NO_MUN) = upper(cn.municipio)
 AND upper(m.SG_UF) = upper(cn.uf)
 join df_uf u
  ON u.sg_uf = m.SG_UF
"""

In [0]:
df_final = spark.sql(query)

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS database_name")

In [0]:
%sql

drop table hive_metastore.dimensao.dm_municipio

In [0]:
save_hive_table(df_final, target_path, pk)